# Math toolkit

**Prerequisites:** complete `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, and `01_foundations/market_data_and_curves.ipynb` in this curriculum (foundations track). This notebook assumes you are comfortable with `finstack_quant` core types and imports used there.

**In this notebook:** Rust-backed numerics in `finstack_quant.core.math` — linear algebra (Cholesky), descriptive statistics, distribution helpers, and compensated summation — plus a simulation that ties several pieces together.

## Mathematical building blocks

Quant code repeatedly needs a small set of primitives:

- **Symmetric positive-definite (SPD) linear systems** — Cholesky factorization \(A = LL^\top\) and triangular solves appear in calibration, covariance work, and Monte Carlo correlation sampling.
- **Correlation matrices** — must be symmetric, with ones on the diagonal, entries in \([-1,1]\), and positive semi-definite. Invalid matrices break downstream pricing and risk.
- **Robust descriptive statistics** — means, variances, covariances, and quantiles for exploratory checks and validation.
- **Special functions** — Gaussian CDF/PDF and related tools for models and transforms.
- **Stable summation** — naive `sum()` can lose tiny terms when magnitudes differ wildly; compensated methods reduce rounding error.

The APIs below are thin Python bindings over `finstack-quant-core` implementations, so you get consistent numerics across the stack.

### Linear algebra: Cholesky and correlation checks

`cholesky_decomposition` returns a lower-triangular \(L\) with \(A = LL^\top\). `cholesky_solve` solves \(Ax=b\) via **forward** substitution on \(Ly=b\) and **backward** substitution on \(L^\top x=y\) using that Cholesky factor \(L\). `validate_correlation_matrix` lives in `finstack_quant.models.correlation` and takes **flat row-major** storage `(values, n)` (handy when matrices are packed in buffers).

In [ ]:
from finstack_quant.core.math import linalg
from finstack_quant.core.math.linalg import cholesky_decomposition, cholesky_solve
from finstack_quant.models.correlation import validate_correlation_matrix

L = cholesky_decomposition([[4.0, 2.0], [2.0, 3.0]])
print("Cholesky L:", L)

x = cholesky_solve(L, [1.0, 1.0])
print("Solution:", x)

validate_correlation_matrix([1.0, 0.5, 0.5, 1.0], 2)
print("validate_correlation_matrix (flat row-major, n=2): OK")

print("SINGULAR_THRESHOLD:", linalg.SINGULAR_THRESHOLD)
print("DIAGONAL_TOLERANCE:", linalg.DIAGONAL_TOLERANCE)

### Statistics: moments and association

Sample variance uses the \(n-1\) denominator; `population_variance` uses \(n\). `quantile` follows the R-7 / NumPy default with linear interpolation.

### Consecutive positive runs

`longest_positive_run` is useful for covenant tests, drawdown streak checks, and any rule that cares about the longest positive run in a numeric series.

In [ ]:
from finstack_quant.core.math import longest_positive_run

streak = longest_positive_run([1.0, 2.0, -1.0, 3.0, 4.0, 5.0, 0.0])
print("longest strictly positive run:", streak)

In [ ]:
from finstack_quant.core.math import stats

data = [1.0, 2.0, 3.0, 4.0, 5.0]
print("Mean:", stats.mean(data))
print("Variance:", stats.variance(data))
print("Pop variance:", stats.population_variance(data))

x = [1.0, 2.0, 3.0, 4.0, 5.0]
y = [2.0, 4.0, 5.0, 4.0, 5.0]
print("Correlation:", stats.correlation(x, y))
print("Covariance:", stats.covariance(x, y))
print("Quantile 50%:", stats.quantile(data, 0.5))

### Utilities: `longest_positive_run`

`longest_positive_run(values)` returns the length of the longest run of values satisfying the (hard-coded) predicate "strictly positive". Handy for simple streak / run-length checks on return series.

In [ ]:
from finstack_quant.core.math import longest_positive_run

returns = [0.01, 0.02, -0.005, 0.03, 0.01, 0.0, 0.04]
print("returns:", returns)
print("longest positive streak:", longest_positive_run(returns))

print("empty:", longest_positive_run([]))
print("all negative:", longest_positive_run([-1.0, -2.0]))

### Special functions: Gaussian and gamma-related helpers

Standard normal CDF \(\Phi\), PDF \(\varphi\), inverse CDF, `erf`, and \(\ln\Gamma\) for common pricing and statistics workflows.

In [ ]:
from finstack_quant.core.math.special_functions import norm_cdf, norm_pdf, standard_normal_inv_cdf, erf, ln_gamma

print("N(0):", norm_cdf(0.0))
print("N(1.96):", norm_cdf(1.96))
print("phi(0):", norm_pdf(0.0))
print("inv_N(0.975):", standard_normal_inv_cdf(0.975))
print("erf(1):", erf(1.0))
print("ln_gamma(5):", ln_gamma(5.0))

### Compensated summation: Kahan and Neumaier

When many small terms combine with large partial sums, floating-point rounding can dominate. **Kahan** is strong when signs are similar; **Neumaier** tends to behave better with mixed signs (e.g., cashflows).

In [ ]:
from finstack_quant.core.math.summation import kahan_sum, neumaier_sum

values = [1.0] + [1e-16] * 10000 + [-1.0]
naive = sum(values)
kahan = kahan_sum(values)
neumaier = neumaier_sum(values)
expected = 1e-16 * 10000
print(f"Naive sum:    {naive}")
print(f"Kahan sum:    {kahan}")
print(f"Neumaier sum: {neumaier}")
print(f"Expected:     {expected}")

## Mini-example: correlated normals and numerical stability

1. Choose a valid \(3 \times 3\) **correlation** matrix \(C\).
2. Factor \(C = LL^\top\) with `cholesky_decomposition`.
3. Draw independent standard normals \(Z\) (`random` module, fixed seed).
4. Set \(X = LZ\) with `linalg.apply_lower_triangular(L, z)` — the Rust triangular matrix-vector product, which takes \(L\) as a list of **rows**, exactly as `cholesky_decomposition` returns it.
5. Compare **sample** pairwise correlations from many draws to \(C\) using `stats.correlation`.

Steps 2, 4 and 5 all run in Rust; only the iid draw stays in Python, as fixture data. Step 5 is a deliberate cross-check: it verifies the Rust factorization and apply step reproduce the target correlation.

A second short experiment shows **compensated summation** when a huge positive and negative term nearly cancel: the exact residual is moderate, but a **left-to-right** accumulation (typical `total += x` loops) can lose it to rounding. The naive loop there is written out on purpose — the contrast against `kahan_sum` / `neumaier_sum` is the lesson. Note that Python's built-in `sum()` applies extra care for floats, so we compare against that too.

In [ ]:
import random

from finstack_quant.core.math.linalg import apply_lower_triangular, cholesky_decomposition
from finstack_quant.core.math import stats
from finstack_quant.models.correlation import validate_correlation_matrix

C = [
    [1.0, 0.7, 0.5],
    [0.7, 1.0, 0.3],
    [0.5, 0.3, 1.0],
]
validate_correlation_matrix([v for row in C for v in row], len(C))
print("Target correlation matrix validated: OK")

L = cholesky_decomposition(C)
print("Cholesky L (first row):", L[0])

# The Cholesky-apply step X = L Z is a Rust primitive: `apply_lower_triangular`
# takes L as a list of ROWS (the same shape `cholesky_decomposition` returns)
# and the iid draw z, and returns the correlated vector. The iid normals below
# stay in Python on purpose — this is fixture generation, not financial logic.
random.seed(42)
n_samples = 40_000
c0, c1, c2 = [], [], []
for _ in range(n_samples):
    z = [random.gauss(0.0, 1.0) for _ in range(3)]
    x = apply_lower_triangular(L, z)
    c0.append(x[0])
    c1.append(x[1])
    c2.append(x[2])

# Cross-check (pedagogical): sample correlations computed by the Rust
# stats.correlation must converge to the target matrix C. This verifies the
# Rust factorization and the Rust apply step against the theory, end to end.
print("Pair (0,1) target:", C[0][1], "sample corr:", stats.correlation(c0, c1))
print("Pair (0,2) target:", C[0][2], "sample corr:", stats.correlation(c0, c2))
print("Pair (1,2) target:", C[1][2], "sample corr:", stats.correlation(c1, c2))

In [ ]:
from finstack_quant.core.math.summation import kahan_sum, neumaier_sum


def naive_left_to_right(values):
    """Strict left-to-right float add (typical hand-written loop semantics)."""
    total = 0.0
    for v in values:
        total += v
    return total


# Large-magnitude cancellation: left-to-right IEEE addition can drop moderate terms.
values = [1e16] + [1.0] * 1000 + [-1e16]
builtin_sum = sum(values)
naive_cancel = naive_left_to_right(values)
kahan_cancel = kahan_sum(values)
neumaier_cancel = neumaier_sum(values)
print("Sum of [1e16, +1 (x1000), -1e16] — exact total should be 1000:")
print("  builtin sum():", builtin_sum)
print("  left-to-right:", naive_cancel)
print("  kahan:         ", kahan_cancel)
print("  neumaier:      ", neumaier_cancel)
print("  expected:      ", 1000.0)

## Takeaways

- **`finstack_quant.core.math.linalg`** — Cholesky factorization, triangular solves for SPD systems, and `apply_lower_triangular(L_rows, z)` for the \(LZ\) product.
- **`finstack_quant.models.correlation`** — `validate_correlation_matrix(values, n)` enforces the standard correlation-matrix constraints on **flat row-major** packed matrices, which matches patterns in benchmarks and low-level pipelines.
- **`finstack_quant.core.math.stats`** — means, variances, covariance, correlation, and quantiles for quick sanity checks on data and simulations.
- **Special functions** — Gaussian and gamma-related primitives for models that depend on \(\Phi\), \(\varphi\), or \(\ln\Gamma\).
- **Kahan / Neumaier** — use when summing long sequences where rounding error matters; Neumaier is often preferable when signs mix.
- **Correlated normals** — if \(Z\) is standard independent and \(C = LL^\top\), then \(LZ\) has correlation \(C\); sample correlations converge to targets as you increase paths.

**Next:** continue with `01_foundations/market_data_and_curves.ipynb`, `02_pricing/pricing_fundamentals.ipynb`, or `07_advanced_quant/monte_carlo_simulation.ipynb` depending on your track.